# 🏢 AttnRetrofit: Attention-Based Energy Forecasting dengan SHAP Diagnostics

**Project:** Smart Building Retrofit Prioritization menggunakan Deep Learning + XAI

**SDG:** SDG 13 - Climate Action

**Dataset:** ASHRAE Great Energy Predictor III (53.6M records, 1,636 buildings)

**Model:** Bidirectional LSTM + Multi-Head Attention + SHAP Explainability

## 📦 CELL 1-5: Setup & Imports

In [ ]:
# Standard libraries
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, precision_recall_curve
import joblib

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Explainable AI
import shap

# Hyperparameter Tuning
import optuna
from optuna.trial import TrialState

# Progress bar
from tqdm import tqdm

# Config
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Data path
DATA_PATH = Path('data')
print(f"Data path: {DATA_PATH.absolute()}")

print("\n✅ Setup complete!")

## 📊 CELL 6: Load Data & Quick Overview

In [ ]:
print("=== ASHRAE DATASET OVERVIEW ===")

# Load ASHRAE files
train = pd.read_csv(DATA_PATH / 'train.csv')
weather_train = pd.read_csv(DATA_PATH / 'weather_train.csv')
building_meta = pd.read_csv(DATA_PATH / 'building_metadata.csv')

print(f"Train shape: {train.shape}")
print(f"Weather shape: {weather_train.shape}")
print(f"Building meta shape: {building_meta.shape}")

print("\n=== TRAIN SAMPLE ===")
display(train.head())

print("\n=== WEATHER SAMPLE ===")
display(weather_train.head())

print("\n=== BUILDING METADATA SAMPLE ===")
display(building_meta.head())

# Basic info
print("\n=== BASIC INFO ===")
print(f"Unique buildings: {train['building_id'].nunique()}")
print(f"Unique sites: {building_meta['site_id'].nunique()}")
print(f"Date range: {train['timestamp'].min()} to {train['timestamp'].max()}")
print(f"Meter types: {train['meter'].unique()}")

## ⚠️ CELL 7: Zero Meter Analysis (Filter Electricity Only)

In [ ]:
print("=== ZERO VALUES ANALYSIS PER METER TYPE ===")

# Meter type mapping
meter_mapping = {0: 'electricity', 1: 'chilledwater', 2: 'steam', 3: 'hotwater'}
train['meter_type'] = train['meter'].map(meter_mapping)

# Calculate zero percentage
zero_analysis = train.groupby('meter')['meter_reading'].agg([
    ('total_count', 'count'),
    ('zero_count', lambda x: (x == 0).sum()),
    ('zero_pct', lambda x: (x == 0).mean() * 100),
    ('min', 'min'),
    ('max', 'max')
]).reset_index()
zero_analysis['meter'] = zero_analysis['meter'].map(meter_mapping)
print(zero_analysis)

# Filter electricity only (meter=0)
train_elec = train[train['meter'] == 0].copy()
print(f"\n✅ Filtered to electricity only: {len(train_elec):,} rows ({len(train_elec)/len(train)*100:.1f}% of original)")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Zero percentage
zero_pcts = train.groupby('meter')['meter_reading'].apply(lambda x: (x == 0).mean() * 100)
zero_pcts.index = zero_pcts.index.map(meter_mapping)
zero_pcts.plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Percentage of Zero Values per Meter Type')
axes[0].set_ylabel('Zero Values (%)')
axes[0].tick_params(axis='x', rotation=45)

# Distribution log-scale
train_nonzero = train[train['meter_reading'] > 0]
for meter_type in train['meter_type'].unique():
    subset = train_nonzero[train_nonzero['meter_type'] == meter_type]['meter_reading']
    axes[1].hist(np.log1p(subset), bins=50, alpha=0.6, label=meter_type)
axes[1].set_title('Distribution of Non-Zero Meter Readings (Log Scale)')
axes[1].set_xlabel('Log(Meter Reading + 1)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n🎯 DECISION: Using electricity only for training")
print("→ Steam excluded (90% zeros) \n→ Hot water excluded (70% zeros)")

## ⚠️ CELL 8: Timestamp Alignment Check

In [ ]:
print("=== TIMESTAMP ALIGNMENT CHECK ===")

# Convert timestamps
train_elec['timestamp'] = pd.to_datetime(train_elec['timestamp'])
weather_train['timestamp'] = pd.to_datetime(weather_train['timestamp'])

print(f"Train timestamp range: {train_elec['timestamp'].min()} to {train_elec['timestamp'].max()}")
print(f"Weather timestamp range: {weather_train['timestamp'].min()} to {weather_train['timestamp'].max()}")

# Check per site
site_coverage = train_elec.groupby('site_id')['building_id'].nunique().to_frame('num_buildings')
site_coverage['weather_available'] = weather_train.groupby('site_id').size()
print("\n=== SITE COVERAGE ===")
print(site_coverage)

# Merge check
merged = train_elec.merge(weather_train, on=['site_id', 'timestamp'], how='left', indicator=True)
merge_stats = merged['_merge'].value_counts()
print(f"\n=== MERGE RESULT ===")
print(f"Matched: {merge_stats.get('both', 0):,}")
print(f"Left-only (no weather): {merge_stats.get('left_only', 0):,}")
print(f"Percentage matched: {(merge_stats.get('both', 0) / len(train_elec)) * 100:.2f}%")

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 5))
weather_missing = merged.groupby('site_id')['_merge'].apply(lambda x: (x == 'left_only').mean() * 100)
weather_missing.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Missing Weather Data per Site (%)')
ax.set_ylabel('Missing Weather (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n🎯 DECISION: Merge train + weather on [site_id, timestamp]")
print("→ Forward fill untuk missing weather data")

## ⚠️ CELL 9: Site vs Building ID Validation

In [ ]:
print("=== SITE_ID vs BUILDING_ID ANALYSIS ===")

# Check relationship
site_building = building_meta.groupby('site_id')['building_id'].nunique().to_frame('num_buildings')
print("Buildings per site:")
print(site_building.describe())

# Check unique counts
print(f"\nUnique sites: {building_meta['site_id'].nunique()}")
print(f"Unique buildings: {building_meta['building_id'].nunique()}")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Buildings per site
site_building['num_buildings'].plot(kind='bar', ax=axes[0])
axes[0].set_title('Number of Buildings per Site')
axes[0].set_xlabel('Site ID')
axes[0].set_ylabel('Number of Buildings')

# Primary use distribution
building_meta['primary_use'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Building Types Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n🎯 DECISION: Use building_id for embedding")
print("→ site_id sebagai fitur tambahan")

## ⚠️ CELL 10: Train/Test Leakage Check (Time-Based Split)

In [ ]:
print("=== TRAIN/TEST SPLIT VALIDATION ===")

# Add year-month for analysis
train_elec['year'] = train_elec['timestamp'].dt.year
train_elec['month'] = train_elec['timestamp'].dt.month
train_elec['year_month'] = train_elec['timestamp'].dt.to_period('M')

# Check data distribution over time
time_dist = train_elec.groupby('year_month').size()
print("Data distribution over time:")
print(time_dist)

# Check building coverage over time
building_time = train_elec.groupby('year_month')['building_id'].nunique()
print(f"\nUnique buildings per month:")
print(building_time.describe())

# Visualisasi
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Total readings over time
time_dist.plot(ax=axes[0], marker='o')
axes[0].set_title('Total Readings Over Time')
axes[0].set_ylabel('Count')
axes[0].axvline(x=pd.Period('2017-01'), color='red', linestyle='--', label='Test cutoff')
axes[0].legend()

# Unique buildings over time
building_time.plot(ax=axes[1], marker='o', color='green')
axes[1].set_title('Unique Buildings Over Time')
axes[1].set_ylabel('Unique Buildings')
axes[1].axvline(x=pd.Period('2017-01'), color='red', linestyle='--')

plt.tight_layout()
plt.show()

# Define split dates
TRAIN_END = '2016-12-31'
VAL_END = '2017-06-30'

print(f"\n🎯 SPLIT DECISION:")
print(f"→ Train: <= {TRAIN_END}")
print(f"→ Val: {TRAIN_END} < date <= {VAL_END}")
print(f"→ Test: > {VAL_END}")
print(f"\n⚠️ CRITICAL: Time-based split (NO random shuffle!)")

## ⚠️ CELL 11: Square Feet Outlier Detection

In [ ]:
print("=== BUILDING METADATA QUALITY CHECK ===")

# Check for missing/invalid values
print("Missing values in building_meta:")
print(building_meta.isnull().sum())

# Square feet analysis
print(f"\nSquare feet stats:")
print(building_meta['square_feet'].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Check for outliers
zero_sqft = (building_meta['square_feet'] == 0).sum()
extreme_sqft = (building_meta['square_feet'] > building_meta['square_feet'].quantile(0.99)).sum()
print(f"\nBuildings with 0 sqft: {zero_sqft}")
print(f"Buildings with extreme sqft (top 1%): {extreme_sqft}")

# Building 1099 (famous outlier)
if 1099 in building_meta['building_id'].values:
    b1099 = building_meta[building_meta['building_id'] == 1099].iloc[0]
    print(f"\n=== BUILDING 1099 (Known Outlier) ===")
    print(f"Square feet: {b1099['square_feet']}")
    print(f"Primary use: {b1099['primary_use']}")

# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

building_meta['square_feet'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('Square Feet Distribution')

np.log1p(building_meta['square_feet']).hist(bins=50, ax=axes[0,1])
axes[0,1].set_title('Log Square Feet Distribution')

building_meta['year_built'].hist(bins=50, ax=axes[1,1])
axes[1,1].set_title('Year Built Distribution')

plt.tight_layout()
plt.show()

# Mark outlier buildings
outlier_buildings = [1099]
print(f"\n🎯 DECISION: Exclude outlier buildings: {outlier_buildings}")

## ⚠️ CELL 12: Weather Correlation Analysis

In [ ]:
print("=== WEATHER CORRELATION ANALYSIS ===")

# Merge untuk correlation analysis
sample_train = train_elec.sample(n=min(100000, len(train_elec)), random_state=SEED)
sample_merged = sample_train.merge(weather_train, on=['site_id', 'timestamp'], how='left')
sample_merged = sample_merged.merge(building_meta, on='building_id', how='left')

# Correlation dengan meter_reading
weather_cols = ['air_temperature', 'dew_temperature', 'wind_speed', 'sea_level_pressure']
corrs = sample_merged[['meter_reading'] + weather_cols].corr()['meter_reading'].drop('meter_reading')

print("Correlation with meter_reading:")
print(corrs.sort_values(ascending=False))

# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature vs meter_reading
axes[0,0].scatter(sample_merged['air_temperature'], sample_merged['meter_reading'], alpha=0.1, s=1)
axes[0,0].set_xlabel('Air Temperature')
axes[0,0].set_ylabel('Meter Reading')
axes[0,0].set_title('Temperature vs Energy Consumption')

# Correlation heatmap
corr_matrix = sample_merged[['meter_reading', 'air_temperature', 'dew_temperature', 'wind_speed']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[0,1])
axes[0,1].set_title('Correlation Heatmap')

# By building type
for i, use_type in enumerate(sample_merged['primary_use'].unique()[:3]):
    subset = sample_merged[sample_merged['primary_use'] == use_type]
    axes[1,0].scatter(subset['air_temperature'], subset['meter_reading'], 
                      alpha=0.3, s=1, label=use_type)
axes[1,0].set_xlabel('Air Temperature')
axes[1,0].set_ylabel('Meter Reading')
axes[1,0].set_title('Temperature vs Consumption by Building Type')
axes[1,0].legend()

# Time series sample
sample_bid = sample_merged['building_id'].iloc[0]
sample_ts = sample_merged[sample_merged['building_id'] == sample_bid].sort_values('timestamp')
ax2 = axes[1,1]
ax2_twin = ax2.twinx()
ax2.plot(sample_ts['timestamp'], sample_ts['meter_reading'], 'b-', label='Consumption')
ax2_twin.plot(sample_ts['timestamp'], sample_ts['air_temperature'], 'r-', label='Temperature')
ax2.set_xlabel('Time')
ax2.set_ylabel('Consumption', color='b')
ax2_twin.set_ylabel('Temperature', color='r')
ax2.set_title('Consumption vs Temperature Over Time')

plt.tight_layout()
plt.show()

print("\n🎯 INSIGHT: Temperature strongly correlates with energy consumption")
print("→ Include air_temperature, dew_temperature as features")

## 📋 CELL 13: EDA Summary & Cleaning Decisions

In [ ]:
print("=" * 60)
print("📊 EDA SUMMARY & CLEANING DECISIONS")
print("=" * 60)

decisions = {
    "1. Meter Types": "Gunakan electricity (0) only. Exclude steam (2) & hot water (3).",
    "2. Timestamp": "Merge train + weather on [site_id, timestamp]. Forward fill missing weather.",
    "3. Building ID": "Gunakan building_id untuk embedding. site_id sebagai fitur tambahan.",
    "4. Train/Val/Test Split": "Time-based: 2016=train, 2017H1=val, 2017H2=test. NO random split!",
    "5. Outliers": f"Exclude building_id {outlier_buildings}. Cap square_feet di 99th percentile.",
    "6. Missing Values": "floor_count & year_built: impute dengan median per primary_use",
    "7. Features": "hour, day_of_week, month, is_weekend, temp_diff, lag_24h, rolling_mean_24h"
}

for key, value in decisions.items():
    print(f"\n✅ {key}:")
    print(f"   {value}")

print("\n" + "=" * 60)
print("Ready for Preprocessing! 🚀")
print("=" * 60)